In [6]:
import pandas as pd

df = pd.read_csv("metabric_after_eda.csv")
df.head()

,PATIENT_ID,LYMPH_NODES_EXAMINED_POSITIVE,NPI,CELLULARITY,CHEMOTHERAPY,COHORT,ER_IHC,HER2_SNP6,HORMONE_THERAPY,INFERRED_MENOPAUSAL_STATE,...,TG,THADA,THSD7A,TP53,UBR5,USH2A,USP28,USP9X,UTRN,recurred
0,MB-0000,10.0,6.044,NaN,NO,1.0,Positive,NEUTRAL,YES,Post,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,MB-0002,0.0,4.020,High,NO,1.0,Positive,NEUTRAL,YES,Pre,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,MB-0005,1.0,4.030,High,YES,1.0,Positive,NEUTRAL,YES,Pre,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,MB-0006,3.0,4.050,Moderate,YES,1.0,Positive,NEUTRAL,YES,Pre,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,MB-0008,8.0,6.080,High,YES,1.0,Positive,NEUTRAL,YES,Post,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
# 1. TUMOR_STAGE = 0 but LYMPH_NODES_EXAMINED_POSITIVE > 0
c1 = df[(df["TUMOR_STAGE"] == 0) & (df["LYMPH_NODES_EXAMINED_POSITIVE"] > 0)]
print(f"Stage 0 but positive lymph nodes: {len(c1)}")

Stage 0 but positive lymph nodes: 0


In [9]:
print(df[df["ER_STATUS"] == "Negative"]["HORMONE_THERAPY"].value_counts())
print(df[df["PR_STATUS"] == "Positive"]["ER_STATUS"].value_counts())

HORMONE_THERAPY
NO     337
YES    137
Name: count, dtype: int64
ER_STATUS
Positive    1020
Negative      20
Name: count, dtype: int64


In [5]:
# 3. GRADE = 1 but NPI > 5 (high risk)
c3 = df[(df["GRADE"] == 1) & (df["NPI"] > 5)]
print(f"Grade 1 but high NPI (>5): {len(c3)}")

Grade 1 but high NPI (>5): 0


In [6]:
# 4. TUMOR_SIZE = 0
c4 = df[df["TUMOR_SIZE"] == 0]
print(f"Tumor size = 0: {len(c4)}")

Tumor size = 0: 0


In [7]:
# 5. TUMOR_STAGE = 4 but LYMPH_NODES_EXAMINED_POSITIVE = 0
c5 = df[(df["TUMOR_STAGE"] == 4) & (df["LYMPH_NODES_EXAMINED_POSITIVE"] == 0)]
print(f"Stage 4 but no positive lymph nodes: {len(c5)}")

Stage 4 but no positive lymph nodes: 3


In [8]:
# 6. AGE < 18
c6 = df[df["AGE_AT_DIAGNOSIS"] < 18]
print(f"Age at diagnosis < 18: {len(c6)}")

Age at diagnosis < 18: 0


In [10]:
c5 = df[(df["TUMOR_STAGE"] == 4) & (df["LYMPH_NODES_EXAMINED_POSITIVE"] == 0)]
print(c5[["PATIENT_ID", "TUMOR_STAGE", "LYMPH_NODES_EXAMINED_POSITIVE", 
          "TUMOR_SIZE", "ER_STATUS", "recurred"]])

    PATIENT_ID  TUMOR_STAGE  LYMPH_NODES_EXAMINED_POSITIVE  TUMOR_SIZE  \
5      MB-0010          4.0                            0.0        31.0   
12     MB-0036          4.0                            0.0        22.0   
174    MB-0268          4.0                            0.0        19.0   

    ER_STATUS  recurred  
5    Positive       1.0  
12   Positive       1.0  
174  Positive       1.0  


None of these are definitive errors , they're all medically explainable

In [ ]:
# 2. ER_STATUS vs ER_IHC disagreement
c2 = df[(df["ER_STATUS"] == "Positive") & (df["ER_IHC"] == "Negative") |
        (df["ER_STATUS"] == "Negative") & (df["ER_IHC"] == "Positve")]
print(f"ER_STATUS vs ER_IHC disagreement: {len(c2)}")

ER_STATUS vs ER_IHC disagreement: 128


These two columns measure the same thing (estrogen receptor status) but come from different sources or time points. 128 patients have conflicting values between them. This is a data quality issue

In [7]:
# 3. CLAUDIN_SUBTYPE = Basal but ER_STATUS = Positive
# Basal subtype is almost always ER negative
c3 = df[(df["CLAUDIN_SUBTYPE"] == "Basal") & (df["ER_IHC"] == "Positive")]
print(f"Basal subtype but ER Positive: {len(c3)}")

Basal subtype but ER Positive: 33


Basal subtype is defined as triple negative (ER-, PR-, HER2-) by convention. Having ER+ Basal is a contradiction by definition

In [14]:
# 4. CLAUDIN_SUBTYPE = LumA but GRADE = 3
# LumA is typically low grade
c4 = df[(df["CLAUDIN_SUBTYPE"] == "LumA") & (df["GRADE"] == 3)]
print(f"LumA subtype but Grade 3: {len(c4)}")

LumA subtype but Grade 3: 172


LumA is defined as low proliferation, low grade, ER+ tumor. Grade 3 means highly aggressive and fast growing — the opposite of LumA characteristics. These 172 patients are likely misclassified

In [15]:
# 5. HER2_STATUS = Positive but CLAUDIN_SUBTYPE = LumA
# LumA is typically HER2 negative
c5 = df[(df["HER2_STATUS"] == "Positive") & (df["CLAUDIN_SUBTYPE"] == "LumA")]
print(f"HER2 Positive but LumA subtype: {len(c5)}")

HER2 Positive but LumA subtype: 21


LumA tumors are by definition HER2 negative. Having HER2+ LumA is a contradiction

In [17]:
# 1. NPI range check — NPI should be between 1 and 8 clinically
c1 = df[(df["NPI"] < 1) | (df["NPI"] > 8)]
print(f"NPI out of range (1-8): {len(c1)}")

NPI out of range (1-8): 0


In [18]:
# 2. Negative tumor size
c2 = df[df["TUMOR_SIZE"] < 0]
print(f"Negative tumor size: {len(c2)}")

Negative tumor size: 0


In [19]:
# 3. Negative lymph nodes
c3 = df[df["LYMPH_NODES_EXAMINED_POSITIVE"] < 0]
print(f"Negative lymph node count: {len(c3)}")

Negative lymph node count: 0


In [24]:
print(df.dtypes.to_string())

PATIENT_ID                           str
LYMPH_NODES_EXAMINED_POSITIVE    float64
NPI                              float64
CELLULARITY                          str
CHEMOTHERAPY                         str
COHORT                           float64
ER_IHC                               str
HER2_SNP6                            str
HORMONE_THERAPY                      str
INFERRED_MENOPAUSAL_STATE            str
SEX                                  str
INTCLUST                             str
AGE_AT_DIAGNOSIS                 float64
OS_MONTHS                        float64
OS_STATUS                            str
CLAUDIN_SUBTYPE                      str
THREEGENE                            str
VITAL_STATUS                         str
LATERALITY                           str
RADIO_THERAPY                        str
HISTOLOGICAL_SUBTYPE                 str
BREAST_SURGERY                       str
RFS_MONTHS                       float64
RFS_STATUS                           str
SAMPLE_ID       

Should be integer:

GRADE should be int (1, 2, 3), only has NaN keeping it float

TUMOR_STAGE should be int (0, 1, 2, 3, 4)

LYMPH_NODES_EXAMINED_POSITIVE should be int (count of nodes)


Should be dropped (wrong to keep as features):

OS_MONTHS, OS_STATUS, VITAL_STATUS : outcome leakage
RFS_MONTHS, RFS_STATUS : outcome leakage
PATIENT_ID, SAMPLE_ID : IDs
SEX, CANCER_TYPE, SAMPLE_TYPE : no variance

Redundant features to remove :

RFS_STATUS:recurred is a feature created previously in EDA step , it's a cleaned numeric version of RFS_STATUS

ER_IHC  is the same as ER_STATUS

ONCOTREE_CODE is the same as CANCER_TYPE_DETAILED


ER_IHC comes from data_clinical_patient.txt



ER_STATUS comes from data_clinical_sample.txt



Keeping ER_STATUS is better because sample-level measurement is more directly tied to the tumor tissue tested